## Item A

In [8]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import BloodMNIST

# Definir a transformação para converter as imagens em Tensores PyTorch [0.0, 1.0]
# Nota: O ToTensor() já divide automaticamente os pixels [0, 255] por 255.0
data_transform = transforms.Compose([
    transforms.ToTensor()
])

# Carregar os datasets oficiais do BloodMNIST (Treino, Validação e Teste)
# O parâmetro download=True baixa a base automaticamente caso não exista localmente
train_dataset = BloodMNIST(split='train', transform=data_transform, download=True)
val_dataset   = BloodMNIST(split='val', transform=data_transform, download=True)
#test_dataset  = BloodMNIST(split='test', transform=data_transform, download=True)

# Criar os DataLoaders para gerenciar o mini-batching
#BATCH_SIZE = 64  # Ou 128
BATCH_SIZE = 256

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
#test_loader  = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# gpu?
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
import ml3.mlp_a as mlp_a
import pandas as pd

#hidden_dims=[128, 256, 512]
#lrs=[1e-3, 1e-4]
#dropouts=[0.0, 0.2, 0.5]

hidden_dims=[256, 512]
lrs=[1e-3, 1e-4]
dropouts=[0.0, 0.2, 0.3]

results = mlp_a.fit_mlp(
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    hidden_dims=hidden_dims,
    lrs=lrs,
    dropouts=dropouts,
    save_tag="_mlp_item_a"
)

df_results = pd.DataFrame(results)


--- Testando: hidden_dim=256, lr=0.001, dropout=0.0 ---

--- Testando: hidden_dim=256, lr=0.001, dropout=0.2 ---

--- Testando: hidden_dim=256, lr=0.001, dropout=0.3 ---

--- Testando: hidden_dim=256, lr=0.0001, dropout=0.0 ---

--- Testando: hidden_dim=256, lr=0.0001, dropout=0.2 ---

--- Testando: hidden_dim=256, lr=0.0001, dropout=0.3 ---

--- Testando: hidden_dim=512, lr=0.001, dropout=0.0 ---

--- Testando: hidden_dim=512, lr=0.001, dropout=0.2 ---

--- Testando: hidden_dim=512, lr=0.001, dropout=0.3 ---

--- Testando: hidden_dim=512, lr=0.0001, dropout=0.0 ---

--- Testando: hidden_dim=512, lr=0.0001, dropout=0.2 ---

--- Testando: hidden_dim=512, lr=0.0001, dropout=0.3 ---

Melhor Configuração Encontrada: {'hidden_dim': 256, 'lr': 0.001, 'dropout': 0.0}
Acurácia Balanceada na Validação: 0.7935


In [10]:
import plotly.express as px
import importlib
importlib.reload(pd)

# Ordenar pelos melhores resultados de Acurácia de Validação
df_sorted = df_results.sort_values(by="val_acc", ascending=False).reset_index(drop=True)

# Aplicar o estilo com gradiente de cor
styled_df = df_sorted.style.background_gradient(
    cmap="Blues", subset=["val_acc"]
).format({
    "val_acc": "{:.4f}",
    "lr": "{:.4f}",
    "dropout": "{:.1f}"
}).set_caption("<b>Desempenho da MLP na Busca por Hiperparâmetros</b>")

# Exibir a tabela no Notebook
styled_df

,hidden_dim,lr,dropout,val_acc
0,256,0.0010,0.0,0.7935
1,512,0.0010,0.0,0.7832
2,512,0.0010,0.2,0.7831
3,256,0.0010,0.2,0.7765
4,256,0.0010,0.3,0.7711
5,512,0.0010,0.3,0.7591
6,512,0.0001,0.0,0.7317
7,512,0.0001,0.3,0.7308
8,256,0.0001,0.0,0.7202
9,512,0.0001,0.2,0.7160


In [11]:
# Criar o gráfico de coordenadas paralelas
fig = px.parallel_coordinates(
    df_results,
    dimensions=['hidden_dim', 'lr', 'dropout', 'val_acc'],
    color='val_acc',
    color_continuous_scale=px.colors.diverging.Tealrose,
    labels={
        'hidden_dim': 'Neurônios Ocultos',
        'lr': 'Taxa de Aprendizado (LR)',
        'dropout': 'Dropout (p)',
        'val_acc': 'Acurácia Bal. (Val)'
    },
    title="Exploração de Hiperparâmetros da MLP - Coordenadas Paralelas"
)

# Ajustar layout para exibição no Notebook
fig.update_layout(template="plotly_white")
fig.show()

# Linhas mais claras/vivas: Representam os experimentos com maior acurácia de validação.
# Trajetória: Permite identificar visualmente se uma determinada escolha (por exemplo,
#   lr = 0.001 vs 0.0001 ou dropout = 0.2) concentra as linhas que atingem as maiores
#   pontuações no eixo da direita (val_acc).